# S0 - setup, pin, verify

Quick session (~2-5 min), no GPU generation. The **240-270 min** wall-clock target applies to the work sessions S1-S5, not this one. Run this at the start of every T4 session's VM before its notebook.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'   # exact commit this session runs against

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit
# loads (see 04b). Matches colab_unified_{analysis,training}.ipynb.
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Persistent storage (results/ + HF cache bound to Drive)

In [ ]:
import os, shutil
from pathlib import Path

# One Drive root holds everything that must survive a Colab disconnect / fresh
# VM: results/ (activations, directions, behavioural output, shard checkpoints)
# and the HF weight cache. Change DRIVE_ROOT only to run independent attempts
# side by side. Mirrors colab_unified_analysis.ipynb's persistence plumbing.
DRIVE_ROOT = Path('/content/drive/MyDrive/dpo_v2')
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_HF_CACHE = DRIVE_ROOT / 'hf_cache'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_HF_CACHE.mkdir(parents=True, exist_ok=True)

# Must be set before any transformers/peft import so the ~3 GB base model +
# LoRA adapters download ONCE total, not once per session.
os.environ['HF_HOME'] = str(DRIVE_HF_CACHE)

local_results = Path('results')
if local_results.is_symlink():
    pass  # already wired (cell re-run mid-session)
else:
    if not (DRIVE_RESULTS / 'activations').exists() and local_results.exists():
        # first use of this Drive root: seed it with the checkout's committed
        # results/ so v2 output layers on top instead of starting empty
        shutil.copytree(local_results, DRIVE_RESULTS, dirs_exist_ok=True)
    if local_results.exists():
        shutil.rmtree(local_results)
    local_results.symlink_to(DRIVE_RESULTS, target_is_directory=True)

print('results/ ->', local_results.resolve())
print('HF_HOME  ->', os.environ['HF_HOME'])
!python -m src.analysis.v2_pipeline status

## 5. Benchmark, split-manifest, and gate verification

In [ ]:
import json
latest = json.load(open('data/frozen_v2/LATEST_BENCHMARK.json'))
bench = latest['benchmark_path']
subprocess.run(['python', '-m', 'src.create_direction_split_manifest',
                '--benchmark', bench], check=True)
subprocess.run(['python', '-m', 'src.validate_benchmark_v2',
                '--benchmark', bench,
                '--review-csv', 'data/review/c_review_queue.csv',
                '--gate-config', 'logs/benchmark_gate_config.json',
                '--split-manifest', 'logs/direction_split_manifest.json'], check=True)
from src.analysis.v2_pipeline import STATIC_GATE_FIELDS
status = json.load(open('logs/benchmark_validation_status.json'))
assert all(status.get(k) is True for k in STATIC_GATE_FIELDS), status
print('static gate checks passed:', STATIC_GATE_FIELDS)
from src.v2_io import load_run_inputs
print('load_run_inputs:', load_run_inputs())
print("artifact_freshness_pass:", status['artifact_freshness_pass'],
      '(expected False until this session generates fresh activations)')

## 6. Focused test gate

In [ ]:
V2_TEST_SCOPE = [
  'tests/test_v2_binding_guard.py', 'tests/test_v2_io_binding_contracts.py',
  'tests/analysis/test_verify_activations.py', 'tests/analysis/test_intervention_conditions.py',
]
!python -m pytest {' '.join(V2_TEST_SCOPE)} -q

## 7. Current progress

In [ ]:
!python -m src.analysis.v2_pipeline status